# Côte d'Ivoire - PROMIRE project / UN-REDD

1 · Computing the 2021-2025 activity from the land cover interpreted in CEO
-------
This notebook assigns an activity to each sample by comparing the 2021 and 2025 land cover classes interpreted in Collect Earth Online (CEO), at levels 1 and 2 of the national nomenclature. The criteria follow those of the FREL 2024 (*NERF* in French), complemented by a transition matrix for the changes between forest classes.

| Activity | Meaning |
|:--|:--|
| SNF | stable non-forest |
| SF | stable forest |
| SDegF | stable degraded forest (secondary forest or agroforest in 2021 and in 2025) |
| Deg | degradation |
| Def | deforestation |
| Gain | gain (non-forest → forest, or forest → forest of a higher rank) |

**Inputs** (`data/` folder): the CEO export of the interpreted samples, the stratum of each sample and the boundary of the ERP (Emission Reductions Program, *PRE* in French).

**Output**: `results/CIV_activity_samples_2021_2025_en.csv`, used by notebooks `2` and `3`: one row per sample with `plotid`, `center_lon`, `center_lat`, `strata`, `pre` (sample located in the ERP), `activity`, and the classes of each year.

*English version of `1_CEO_output_formatting.ipynb`: same calculations, same results. The CEO class names (e.g. `Foret_Dense`) and the zone names are data values and are kept in French.*

In [ ]:
import os

import numpy as np
import pandas as pd
import geopandas as gpd

## 1. Reading the data

In [ ]:
points_file   = 'data/ceo-RCI_MONITORING_21-25-plot-data_QA_QC-rev-2.csv'   # CEO export of the interpreted samples
points_strata = 'data/allWithStrata_civ_20_26.shp'                         # stratum of each sample (column mode)
erp_file      = 'data/shapefile_PRE_repaired_geometry.shp'                 # ERP boundary

output_file   = 'results/CIV_activity_samples_2021_2025_en.csv'

initial_year = 2021
final_year   = 2025
years = list(range(initial_year, final_year + 1))

In [ ]:
# Samples interpreted in CEO
df = pd.read_csv(points_file, encoding='latin-1')
print(len(df), 'samples')

### Stratum of each sample

The stratum points are the samples themselves: the nearest point is searched in a metric projection (UTM 30N), and the distance must be zero.

In [ ]:
strata = gpd.read_file(points_strata).rename(columns={'mode': 'strata'})
gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.center_lon, df.center_lat), crs='EPSG:4326')
gdf = gpd.sjoin_nearest(gdf.to_crs('EPSG:32630'), strata[['strata', 'geometry']].to_crs('EPSG:32630'),
                        how='left', distance_col='distance_m')
print('Maximum distance to the stratum point:', gdf['distance_m'].max(), 'm')

df = pd.DataFrame(gdf.drop(columns=['geometry', 'index_right', 'distance_m']))
assert df.plotid.is_unique, 'duplicated plotid'
df.strata.value_counts(dropna=False)

Samples without a stratum (outside the national border or in a masked area of the stratification map) are kept here; the estimation notebooks exclude them.

### Samples located in the ERP

The Emission Reductions Program (ERP) covers part of the territory. Column `pre` is `True` if the sample lies within the ERP polygon (boundary included) and `False` otherwise. Notebooks `2` and `3` use it to produce the ERP estimates.

In [ ]:
erp = gpd.read_file(erp_file).to_crs('EPSG:32630')
points = gpd.GeoSeries(gpd.points_from_xy(df.center_lon, df.center_lat), crs='EPSG:4326').to_crs('EPSG:32630')
df['pre'] = points.intersects(erp.union_all()).to_numpy()
print(df['pre'].sum(), 'samples in the ERP out of', len(df))
pd.crosstab(df['pre'], df['strata'].fillna(0).astype(int).rename('stratum (0 = no stratum)'), margins=True)

## 2. Land cover classes by year and by level

In the CEO file, each class is a column (for example `Occupation_2021:Terres_Forestieres` or `TF_Niveau2_2021:Foret_Dense`) equal to 100 for the class chosen by the interpreter and 0 otherwise. Three tables are built, one per level, with one row per sample and one column per year.

In [ ]:
# Land cover columns (the answers on change, drivers and event years are left out)
lc_columns = [c for c in df.columns
              if '_20' in c and not c.startswith(('Changement', 'Moteur', 'Annee', 'Gain'))]

# Long format: one row per sample and per chosen class
lc = df.melt(id_vars='plotid', value_vars=lc_columns, var_name='column', value_name='value')
lc = lc[lc.value == 100].copy()
lc[['variable', 'class']] = lc.column.str.split(':', expand=True)
lc['year'] = lc.variable.str[-4:].astype(int)
lc['level'] = np.select(
    [lc.variable.str.startswith('Occupation'), lc.variable.str.contains('Niveau2'), lc.variable.str.contains('Niveau3')],
    [1, 2, 3], default=0)
assert (lc.level > 0).all()
lc.head()

In [ ]:
def classes_by_year(level):
    """Samples × years table of the classes of one level (error if a sample has two classes in the same year)."""
    table = lc[lc.level == level].pivot(index='plotid', columns='year', values='class')
    return table.reindex(index=df.plotid, columns=years)


n1 = classes_by_year(1)
n2 = classes_by_year(2)
n3 = classes_by_year(3)
n1.head()

## 3. Samples without a level-1 class

A missing level-1 class, or one equal to `no_data`, makes it impossible to know whether the sample is forest. Only the years 2021 and 2025 are used to compute the activity: a sample without a class in one of these two years receives the activity `ND` and is excluded from the estimates.

In [ ]:
missing = n1.isna() | (n1 == 'no_data')
print('Number of samples without a level-1 class, by year:')
print(missing.sum().to_string())

no_class = missing[missing.any(axis=1)]
listing = pd.DataFrame({
    'plotid': no_class.index,
    'years_without_class': [', '.join(str(y) for y in years if row[y]) for _, row in no_class.iterrows()],
    'activity_ND': (no_class[initial_year] | no_class[final_year]).values,
})
listing

## 4. Computing the activity

Criteria of the FREL 2024, complemented by the 2021-2025 transition matrix, applied to the years 2021 (initial) and 2025 (final). "Forest" = level-1 class `Terres_Forestieres`.

**Changes within the forest.** Each level-2 forest class is linked to a class of the matrix:

| Matrix class | Level-2 classes (CEO) |
|:--|:--|
| 11. Dense forest | `Foret_Dense`, `Foret_Galerie`, `Foret_Sol_Hydromorphe`, `Mangrove` |
| 12. Secondary forest | `Foret_Secondaire_Degradee` |
| 13. Forest plantations | `Plantation_Forestiere_Reboisement` |
| 14. Agroforestry | `Plantation_Agroforestiere` |

Cocoa, including shaded cocoa, remains a cropland (non-forest) class. The matrix ranks the classes 11 > 12 > 13 > 14:

| 2021 ↓ \ 2025 → | 11 | 12 | 13 | 14 |
|:--|:--|:--|:--|:--|
| **11** | SF | Deg | Def | Deg |
| **12** | Gain | SDegF | Def | Deg |
| **13** | Gain | Gain | SF | Deg |
| **14** | Gain | Gain | Gain | SDegF |

A forest that moves to a lower-ranked class is degraded, except towards a forest plantation (deforestation). A forest that moves to a higher-ranked class is a gain. As in the FREL 2024, stable secondary and agroforestry forests are classified SDegF.

**Rules.** They are mutually exclusive:

| Order | Activity | Rule |
|:--|:--|:--|
| 1 | ND | level-1 class missing in 2021 or in 2025 |
| 2 | Def | forest in 2021, non-forest in 2025; or dense / secondary forest in 2021 and forest plantation in 2025 |
| 3 | Deg | forest in 2021 and in 2025, lower-ranked class in 2025 (except forest plantation) |
| 4 | Gain | non-forest in 2021, forest in 2025; or forest in 2021 and in 2025, higher-ranked class in 2025 |
| 5 | SDegF | secondary forest in 2021 and in 2025, or agroforest in 2021 and in 2025 |
| 6 | SF | forest in 2021 and in 2025, same matrix class (11 or 13) |
| 7 | SNF | non-forest in 2021 and in 2025 |

Gain therefore groups two cases: non-forest → forest, and forest → forest of a higher rank. They can be separated in the emissions notebook with `n1_2021`.

In [ ]:
forest = 'Terres_Forestieres'

# Transition-matrix class of each level-2 forest class (rank: 11 > 12 > 13 > 14)
matrix_class = {
    'Foret_Dense': 11, 'Foret_Galerie': 11, 'Foret_Sol_Hydromorphe': 11, 'Mangrove': 11,
    'Foret_Secondaire_Degradee': 12,
    'Plantation_Forestiere_Reboisement': 13,
    'Plantation_Agroforestiere': 14,
}

# The class names must exist in the data (protection against typos)
assert forest in set(n1.stack())
assert set(matrix_class) <= set(n2.stack())

In [ ]:
# 2021 (i = initial) and 2025 (f = final) classes
n1_i, n1_f = n1[initial_year], n1[final_year]
n2_i, n2_f = n2[initial_year], n2[final_year]

forest_i = n1_i == forest
forest_f = n1_f == forest
stays_forest = forest_i & forest_f

# Matrix class (11 to 14) of the forest samples, NaN otherwise
assert set(n2_i[forest_i]) | set(n2_f[forest_f]) <= set(matrix_class), 'forest class without a match in the matrix'
m_i = n2_i.map(matrix_class).where(forest_i)
m_f = n2_f.map(matrix_class).where(forest_f)

missing_class   = missing[initial_year] | missing[final_year]
deforestation   = (forest_i & ~forest_f) | (m_i.isin([11, 12]) & (m_f == 13))
degradation     = stays_forest & (m_f > m_i) & ~deforestation
gain            = (~forest_i & forest_f) | (stays_forest & (m_f < m_i))
stable_degraded = stays_forest & (m_i == m_f) & m_i.isin([12, 14])

conditions = [
    missing_class,
    deforestation,
    degradation,
    gain,
    stable_degraded,
    stays_forest,
    ~forest_i & ~forest_f,
]
activities = ['ND', 'Def', 'Deg', 'Gain', 'SDegF', 'SF', 'SNF']

activity = pd.Series(np.select(conditions, activities, default='Error'), index=n1.index, name='activity')
assert (activity != 'Error').all()
activity.value_counts()

In [ ]:
# Level-2 classes of the forest samples in 2021
n2_i[forest_i].value_counts()

In [ ]:
# Check: activity by level-1 class in 2021
pd.crosstab(n1_i.fillna('(missing)').rename('level 1 in 2021'), activity)

In [ ]:
# Check: activity of the samples that are forest in 2021 and in 2025, by matrix class (compare with the transition matrix)
pd.crosstab([m_i[stays_forest].astype(int).rename('matrix 2021'), m_f[stays_forest].astype(int).rename('matrix 2025')],
            activity[stays_forest])

## 5. Saving

In [ ]:
output = df.set_index('plotid')[['center_lon', 'center_lat', 'strata', 'pre']]
output['activity'] = activity
output = (output.join(n1.add_prefix('n1_'))
                .join(n2.add_prefix('n2_'))
                .join(n3[[initial_year, final_year]].add_prefix('n3_'))
                .reset_index())

assert len(output) == len(df)
os.makedirs(os.path.dirname(output_file), exist_ok=True)
output.to_csv(output_file, index=False)
print(len(output), 'samples saved to', output_file)
output.activity.value_counts()